#  IA agente avec RAG (Génération augmentée par récupération)




## 0) Installation des bibliothèques

On installe tout ce dont on a besoin. `-q` = mode silencieux (peu de logs).
Aucune clé API n'est requise pour la suite.

In [1]:
# Installation des dépendances (à lancer une seule fois par session Colab)
!pip install -q langchain langchain-community faiss-cpu wikipedia transformers accelerate sentencepiece

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 1) Construire le récupérateur (retriever) de la base de connaissances

**But de l'exercice :**
- créer 5 à 8 `Document`, chacun avec un champ `source` (dans `metadata`) ;
- transformer ces documents en vecteurs avec `FakeEmbeddings` ;
- les stocker dans un index **FAISS** ;
- obtenir un `retriever` qui renvoie les **3 documents les plus proches** (top-3).

> ℹ `FakeEmbeddings` génère des vecteurs **aléatoires** (aucun sens sémantique).
> C'est volontaire : ça permet de tester toute la mécanique RAG **sans télécharger**
> de gros modèle d'embeddings. En production, on remplacerait par de vrais embeddings.

In [2]:
# --- Imports nécessaires ---
from langchain_core.documents import Document          # objet "Document" (contenu + métadonnées)
from langchain_community.vectorstores import FAISS      # magasin vectoriel (recherche par similarité)
from langchain_community.embeddings import FakeEmbeddings  # faux embeddings (pour la démo)

# --- Notre mini base de connaissances : 5 documents ---
# Chaque Document a :
#   - page_content : le texte du document ;
#   - metadata["source"] : un identifiant unique, utilisé plus tard pour CITER la source.
kb_docs = [
    Document(
        page_content="""
Agentic systems reason step-by-step about which tools to call instead of invoking tools blindly.
Their core loop is: (1) interpret the user goal, (2) inspect available context, (3) decide whether tools are needed,
(4) call one or more tools in a planned sequence, and (5) synthesize an answer grounded in the tool results.

Key principles for agentic behavior:
- Prefer direct answering from retrieved or provided context when sufficient; avoid unnecessary tool calls.
- Choose the simplest tool or minimal set of tools that can satisfy the user's goal.
- When multiple tools are needed, sketch a short plan (ordered tool calls with dependencies) before executing.
- After each tool call, update an internal scratchpad and re-evaluate whether more tools are still necessary.
- Handle tool failures gracefully: inspect error messages, adjust inputs, or fall back to alternative tools.
""".strip(),
        metadata={"source": "kb:agentic_concept"},  # <-- identifiant de source (pour la citation)
    ),
    Document(
        page_content="""
Retrievers fetch grounding passages from a knowledge base and are the primary interface to internal documents.
Given a user query, the system should: (1) normalize and possibly expand the query, (2) retrieve top-k candidates,
(3) read them carefully, and (4) base the answer primarily on those passages.

Best practices for using retrievers:
- Start with a moderately small k (e.g., 3-10) to keep context focused; increase k only if evidence is clearly missing.
- Reformulate the query if results look noisy or off-topic (e.g., add constraints, synonyms, or key entities).
- Use metadata filters (dates, document types, domains) when available to narrow down to the most relevant sources.
- When multiple passages agree, highlight the shared core facts; when they conflict, surface the disagreement explicitly.
""".strip(),
        metadata={"source": "kb:retrievers"},
    ),
    Document(
        page_content="""
Wikipedia is a broad-coverage, free fallback when the curated knowledge base lacks coverage or appears incomplete.
It should not be the first resort when high-quality internal documents exist, but can complement them for general facts.

Guidelines for using Wikipedia:
- First attempt retrieval over the internal KB; only query Wikipedia if internal retrieval yields little relevant evidence.
- Use Wikipedia mainly for widely accepted, non-controversial information (basic facts, dates, definitions, overviews).
- For critical topics (medical, legal, contentious), treat Wikipedia as a starting point, not the final authority.
- Clearly indicate when information comes from Wikipedia rather than from the internal knowledge base.
""".strip(),
        metadata={"source": "kb:wikipedia_tip"},
    ),
    Document(
        page_content="""
When evidence is thin, ambiguous, or conflicting, the system must be transparent about uncertainty instead of fabricating detail.
Honesty requires separating what is directly supported by sources from what is inferred.

Practical rules for honest behavior:
- Distinguish facts supported by sources, reasonable inferences, and pure speculation.
- If key information is missing, explicitly say the available sources do not fully answer the question.
- Do not invent citations, URLs, names, or numbers to fill gaps.
- When sources conflict, summarize each position and highlight the disagreement.
- Suggest concrete next steps: clarify the question, or expand/narrow the retrieval.
""".strip(),
        metadata={"source": "kb:honesty"},
    ),
    Document(
        page_content="""
Answers should be concise, focused, and well-structured, typically within 2-4 sentences for straightforward questions.
Lead with the main conclusion, then briefly justify it using the most relevant evidence and clearly marked citations.

Style and citation guidelines:
- Directly answer the user's question in the first sentence whenever possible.
- Avoid long preambles or restating the entire question.
- Cite the minimal set of sources that substantively support the key claims, integrated inline near the claims.
- Keep formatting clean: short paragraphs, occasional bullets, no unnecessary verbosity.
""".strip(),
        metadata={"source": "kb:style"},
    ),
]

# --- Construction de l'index vectoriel ---
# 1) On crée les "embeddings" (ici faux, de dimension 256).
embeddings = FakeEmbeddings(size=256)

# 2) On construit l'index FAISS à partir des documents.
vs = FAISS.from_documents(kb_docs, embeddings)

# 3) On en fait un "retriever" qui renverra les 3 documents les plus proches (top-3).
retriever = vs.as_retriever(search_kwargs={"k": 3})

print(" Base de connaissances prête :", len(kb_docs), "documents indexés.")

/tmp/ipykernel_2151/3216495453.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS      # magasin vectoriel (recherche par similarité)


 Base de connaissances prête : 5 documents indexés.


## 2) Outil externe gratuit : recherche Wikipédia

**But de l'exercice :**
- utiliser `langchain_community.utilities` (le `WikipediaAPIWrapper`) pour récupérer des résumés ;
- l'emballer dans une petite fonction `wiki_search` qui renvoie une **liste d'extraits**
  `{titre, résumé}` **+ un éventuel message d'erreur**.

> On enveloppe l'appel réseau dans un `try / except` : si Wikipédia est indisponible
> ou ne renvoie rien, on retourne une liste vide et un message d'erreur au lieu de planter.
> C'est le côté « gérer avec élégance les preuves manquantes ».

In [3]:
from langchain_community.utilities import WikipediaAPIWrapper

# On configure le "wrapper" Wikipédia :
#   - lang="en"                 : langue des articles (anglais, comme les questions de test) ;
#   - top_k_results=2           : au plus 2 articles par recherche ;
#   - doc_content_chars_max=800 : on tronque chaque résumé à 800 caractères (contexte compact).
wiki = WikipediaAPIWrapper(lang="en", top_k_results=2, doc_content_chars_max=800)


def wiki_search(query: str):
    """Cherche sur Wikipédia et renvoie (extraits, erreur).

    Retourne :
      - extraits : liste de dictionnaires {"title": ..., "summary": ...}
      - erreur   : None si tout va bien, sinon une chaîne décrivant le problème.
    """
    try:
        # .load() renvoie une liste d'objets Document (titre dans metadata, texte dans page_content).
        docs = wiki.load(query=query)

        snippets = []
        for doc in docs:
            snippets.append({
                "title": doc.metadata.get("title", "Sans titre"),  # titre de l'article
                "summary": doc.page_content,                        # résumé (déjà tronqué)
            })

        # Si Wikipédia ne trouve rien, on le signale proprement.
        if not snippets:
            return [], "Aucun résultat Wikipédia pour cette requête."

        return snippets, None

    except Exception as e:
        # En cas d'erreur réseau ou autre, on ne plante pas : on renvoie l'erreur.
        return [], f"Erreur lors de la recherche Wikipédia : {e}"


# Petit test : on affiche le titre du 1er extrait trouvé pour "Python programming".
extraits, err = wiki_search("Python programming")
print("Erreur :", err)
if extraits:
    print("1er titre trouvé :", extraits[0]["title"])
    print("Début du résumé  :", extraits[0]["summary"][:120], "...")

Erreur : Erreur lors de la recherche Wikipédia : Expecting value: line 1 column 1 (char 0)


## 3) Planificateur simple (à base de règles)

**But de l'exercice :** décider **où chercher**.
- Si la question contient un **mot-clé connu de la KB** → on privilégie le **retriever** (`action = "kb"`).
- Sinon → on **bascule sur Wikipédia** (`action = "wiki"`).

Le planificateur renvoie un **dictionnaire de plan** tout simple, ex. `{"action": "kb"}`.

In [4]:
# Liste de mots-clés qui "appartiennent" à notre base de connaissances.
# (on met des racines de mots pour attraper les variantes : "transparen" -> transparency/transparent)
kb_keywords = ["agentic", "retriever", "retrieval", "citation", "cite",
               "ground", "transparen", "honest", "uncertain"]


def plan(question: str) -> dict:
    """Décide de la stratégie : 'kb' (base interne) ou 'wiki' (Wikipédia)."""
    question_lower = question.lower()  # on compare en minuscules pour ignorer la casse

    # Si un mot-clé de la KB apparaît dans la question -> on utilise la base interne.
    for keyword in kb_keywords:
        if keyword in question_lower:
            return {"action": "kb", "raison": f"mot-clé KB détecté : '{keyword}'"}

    # Sinon -> repli sur Wikipédia.
    return {"action": "wiki", "raison": "aucun mot-clé KB -> repli Wikipédia"}


# Tests rapides du planificateur :
print(plan("How to ground answers with citations?"))  # -> kb (contient "ground"/"citation")
print(plan("Who created Python?"))                     # -> wiki (aucun mot-clé KB)

{'action': 'kb', 'raison': "mot-clé KB détecté : 'citation'"}
{'action': 'wiki', 'raison': 'aucun mot-clé KB -> repli Wikipédia'}


## 4) Fonction de réponse (stub par défaut, petit modèle HF en option)

**But de l'exercice :**
1. récupérer le contexte **selon le plan** (KB ou Wikipédia) ;
2. fabriquer un prompt qui combine **contexte KB + extraits Wikipédia** ;
3. générer la réponse avec :
   - **par défaut** un **LLM stub** (`FakeListChatModel`) — rapide, aucun téléchargement ;
   - **en option** un **petit modèle Hugging Face** (`sshleifer/tiny-gpt2`) pour tester en local ;
4. **toujours citer** les sources, ex. `[kb:agentic_concept]` ou `[wiki:Python_(programming_language)]` ;
5. si les preuves sont **limitées**, le dire et **proposer une question de suivi**.

>  `sshleifer/tiny-gpt2` est un **modèle minuscule de test** : ses phrases sont
> quasiment du charabia. C'est normal — ici on valide la **mécanique**, pas la qualité.
> Pour la vérification (exercice 5), on utilise le **stub** (`use_tiny_model=False`).

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models.fake import FakeListChatModel

# --- Le gabarit (template) de prompt ---
# On sépare bien les parties avec des retours à la ligne (\n) pour un prompt lisible.
prompt = ChatPromptTemplate.from_template(
    "You are a helpful agentic assistant. Use the given context and wiki snippets.\n"
    "If there is little evidence, say so and suggest a follow-up query.\n"
    "Always cite sources like [kb:agentic_concept] or [wiki:Title].\n\n"
    "Question: {question}\n\n"
    "Context:\n{context}\n\n"
    "Wiki:\n{wiki}\n\n"
    "Answer:"
)


# --- (Option) petit modèle Hugging Face ----------------------------------
# Chargé uniquement si use_tiny_model=True. On importe transformers ici pour
# éviter de le charger quand on utilise seulement le stub.
def get_tiny_generator(model_id: str = "sshleifer/tiny-gpt2"):
    """Charge un pipeline de génération de texte à partir d'un petit modèle HF."""
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)
    return pipeline("text-generation", model=model, tokenizer=tok)


def summarize_with_tiny(prompt_text: str, max_new_tokens: int = 80):
    """Génère une réponse avec le petit modèle HF (sortie approximative)."""
    gen = get_tiny_generator()
    out = gen(
        prompt_text,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        truncation=True,                       # tronque si le prompt dépasse la taille du modèle
        pad_token_id=gen.tokenizer.eos_token_id,  # évite un avertissement (gpt2 n'a pas de pad token)
    )
    full = out[0]["generated_text"]
    # On ne garde que ce que le modèle a AJOUTÉ après le prompt.
    return full[len(prompt_text):].strip()


# --- (Défaut) réponse "stub" ---------------------------------------------
# Le stub ne réfléchit pas : il renvoie une réponse figée. Pour respecter la
# consigne "toujours citer", on construit dynamiquement la liste des sources utilisées.
def stub_answer(kb_sources, wiki_titles):
    citations = [f"[{s}]" for s in kb_sources] + [f"[wiki:{t}]" for t in wiki_titles]
    if not citations:
        # Cas "preuves manquantes" : on le dit et on propose une question de suivi.
        return ("Les sources disponibles ne permettent pas de répondre précisément. "
                "Question de suivi suggérée : pouvez-vous reformuler ou préciser le sujet ?")
    return "Résumé (stub) fondé sur le contexte fourni. Sources : " + " ".join(citations)


def answer_question(question: str, use_tiny_model: bool = False):
    """Répond à une question en suivant le plan, puis cite les sources.

    use_tiny_model=False (défaut) -> réponse via le stub (rapide, sans téléchargement).
    use_tiny_model=True           -> réponse via le petit modèle HF (sshleifer/tiny-gpt2).
    """
    # 1) Décider de la stratégie (KB ou Wikipédia).
    pl = plan(question)

    # 2) Récupérer le contexte correspondant.
    docs = retriever.invoke(question) if pl["action"] == "kb" else []
    wiki_snips, wiki_err = ([], None)
    if pl["action"] == "wiki":
        wiki_snips, wiki_err = wiki_search(question)

    # 3) Mettre en forme le contexte pour le prompt (avec les identifiants de source).
    context_text = "\n".join(
        f"[{d.metadata.get('source')}] {d.page_content}" for d in docs
    ) or "No KB context."
    wiki_text = "\n".join(
        f"[wiki:{s['title']}] {s['summary']}" for s in wiki_snips
    ) or "No wiki snippets."

    # 4) Construire le message final (prompt rempli).
    messages = prompt.format_messages(question=question, context=context_text, wiki=wiki_text)

    # 5) Générer la réponse.
    kb_sources = [d.metadata.get("source") for d in docs]
    wiki_titles = [s.get("title") for s in wiki_snips]

    if use_tiny_model:
        final_answer = summarize_with_tiny(messages[0].content)
    else:
        # On utilise le stub, en lui passant les vraies sources pour la citation.
        stub = FakeListChatModel(responses=[stub_answer(kb_sources, wiki_titles)])
        final_answer = stub.invoke(messages).content

    # 6) Retourner un rapport complet (utile pour la vérification de l'exercice 5).
    return {
        "plan": pl,
        "kb_sources": kb_sources,
        "wiki_sources": wiki_titles,
        "wiki_error": wiki_err,
        "answer": final_answer,
    }

print(" Fonction answer_question() prête (stub par défaut).")

 Fonction answer_question() prête (stub par défaut).


## 5) Vérification rapide sur 3 questions

On teste **3 cas** :
1. une question **couverte par la KB** ;
2. une question **externe** (Wikipédia) ;
3. une question **ambiguë / non couverte** (repli Wikipédia).

Pour chaque question, on affiche : le **plan**, les **sources utilisées** et la **réponse finale**.

> On garde `use_tiny_model=False` pour rester rapide et éviter tout téléchargement.
> Pour essayer le petit modèle HF, mettez `use_tiny_model=True`.

In [6]:
tests = [
    "What are the key principles for agentic behavior?",   # 1) couverte par la KB ("agentic")
    "Who was the 16th President of the United States?",     # 2) externe -> Wikipédia
    "How to make a good cup of coffee?",                    # 3) ambiguë -> repli Wikipédia
]

for q in tests:
    res = answer_question(q, use_tiny_model=False)  # stub = rapide, sans téléchargement
    print("=" * 70)
    print("Question      :", q)
    print("Plan          :", res["plan"])
    print("Sources KB    :", res["kb_sources"])
    print("Sources Wiki  :", res["wiki_sources"])
    if res["wiki_error"]:
        print("Erreur Wiki   :", res["wiki_error"])
    print("Réponse       :", res["answer"])
print("=" * 70)

Question      : What are the key principles for agentic behavior?
Plan          : {'action': 'kb', 'raison': "mot-clé KB détecté : 'agentic'"}
Sources KB    : ['kb:honesty', 'kb:retrievers', 'kb:agentic_concept']
Sources Wiki  : []
Réponse       : Résumé (stub) fondé sur le contexte fourni. Sources : [kb:honesty] [kb:retrievers] [kb:agentic_concept]
Question      : Who was the 16th President of the United States?
Plan          : {'action': 'wiki', 'raison': 'aucun mot-clé KB -> repli Wikipédia'}
Sources KB    : []
Sources Wiki  : []
Erreur Wiki   : Erreur lors de la recherche Wikipédia : Expecting value: line 1 column 1 (char 0)
Réponse       : Les sources disponibles ne permettent pas de répondre précisément. Question de suivi suggérée : pouvez-vous reformuler ou préciser le sujet ?
Question      : How to make a good cup of coffee?
Plan          : {'action': 'wiki', 'raison': 'aucun mot-clé KB -> repli Wikipédia'}
Sources KB    : []
Sources Wiki  : []
Erreur Wiki   : Erreur lors de la